# labelfree-pheno — **self-contained** Kaggle notebook (no GitHub needed)AI4S Open Innovation: AI for Life Science · Category: End-to-End SystemThis notebook embeds the full source code. Set **Settings → Accelerator → GPU T4 x2**, then **Run All** (~1.5 h).All data is public (CC-BY): Caco-2 Virtual-Staining (figshare) + RxRx3-core (Hugging Face).

In [ ]:
import os, sys, subprocess, pathlib
ROOT = pathlib.Path("/content/labelfree-pheno") if pathlib.Path("/content").is_dir() else pathlib.Path("/kaggle/working/labelfree-pheno")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
for d in ["src/labelfree/models", "configs", "scripts", "data", "runs", "report/assets"]:
    (ROOT/d).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT/"src"))
print("workspace:", ROOT)

In [ ]:
%%writefile src/labelfree/__init__.py"""labelfree — label-free phenotyping of organ-on-a-chip microscopy.End-to-end system: bright-field -> fluorescence virtual staining (in silicolabeling) + viability/phenotype scoring + dose-response pharmacology."""__version__ = "0.2.0"

In [ ]:
%%writefile src/labelfree/utils.py
"""Shared utilities: IO, seeds, config, transforms."""
from __future__ import annotations

import random
from pathlib import Path

import numpy as np
import torch
import yaml
from PIL import Image

IMG_EXT = (".jpg", ".jpeg", ".png", ".tif", ".tiff")


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def load_config(path: str | Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def get_device() -> torch.device:
    """TPU (torch_xla) first, then CUDA, then CPU."""
    try:
        import torch_xla.core.xla_model as xm
        dev = xm.xla_device()
        if dev is not None:
            return dev
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def mark_step(device: torch.device) -> None:
    """Flush a lazy PyTorch/XLA step when running on TPU; no-op otherwise."""
    if str(device).startswith("xla"):
        try:
            import torch_xla.core.xla_model as xm
            xm.mark_step()
        except Exception:
            pass


def read_image_gray(path: str | Path) -> np.ndarray:
    """Read a grayscale image as float32 in [0, 1]."""
    img = Image.open(path).convert("L")
    return np.asarray(img, dtype=np.float32) / 255.0


def read_image_rgb(path: str | Path) -> np.ndarray:
    img = Image.open(path).convert("RGB")
    return np.asarray(img, dtype=np.float32) / 255.0


def save_tensor_image(t: torch.Tensor, path: str | Path) -> None:
    """Save a [1,H,W] or [3,H,W] float tensor in [0,1] as uint8 PNG."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    arr = t.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    arr = (arr * 255.0).clip(0, 255).astype(np.uint8)
    if arr.shape[-1] == 1:
        arr = arr[..., 0]
    Image.fromarray(arr).save(path)


def center_crop_np(arr: np.ndarray, size: int) -> np.ndarray:
    h, w = arr.shape[:2]
    top = (h - size) // 2
    left = (w - size) // 2
    return arr[top : top + size, left : left + size]


def ensure_dir(p: str | Path) -> Path:
    p = Path(p)
    p.mkdir(parents=True, exist_ok=True)
    return p


def list_images(folder: str | Path) -> list[Path]:
    folder = Path(folder)
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in IMG_EXT])


In [ ]:
%%writefile src/labelfree/datasets.py"""Datasets for paired bright-field / fluorescence virtual staining."""from __future__ import annotationsimport numpy as npimport pandas as pdimport torchfrom torch.utils.data import Datasetfrom .utils import center_crop_np, read_image_grayclass Caco2PairsDataset(Dataset):    """Paired (BF, green, red) fields from the Caco-2 Virtual-Staining dataset.    pairs.csv columns: id, bf, green, red, split    All image paths are relative to `root` and are grayscale JPEGs.    """    def __init__(self, pairs_csv: str, root: str, split: str = "train",                 crop: int = 192, input_ch: int = 1, target_ch: int = 2):        self.df = pd.read_csv(pairs_csv)        if split != "all":            self.df = self.df[self.df["split"] == split].reset_index(drop=True)        self.root = root        self.crop = crop        self.input_ch = input_ch        self.target_ch = target_ch    def __len__(self) -> int:        return len(self.df)    def _load(self, rel: str) -> np.ndarray:        img = read_image_gray(f"{self.root}/{rel}")        if self.crop and self.crop > 0:            img = center_crop_np(img, self.crop)        return img    def __getitem__(self, idx: int):        row = self.df.iloc[idx]        bf = self._load(row["bf"])        green = self._load(row["green"])        red = self._load(row["red"])        x = torch.from_numpy(bf)[None].float()                       # [1,H,W]        y = torch.from_numpy(np.stack([green, red], 0)).float()      # [2,H,W]        return x, y, row["id"]class SyntheticPairsDataset(Dataset):    """Synthetic paired data for smoke tests / pipeline CI.    BF: blurred blobs + Gaussian noise (nucleus-like dark spots).    Green: viable-cell cytoplasmic signal (soft blobs where nuclei intact).    Red: dead-cell signal (bright ring/blob on a subset of nuclei).    """    def __init__(self, n: int = 64, size: int = 128, seed: int = 0,                 dead_frac: float = 0.3):        self.n = n        self.size = size        rng = np.random.default_rng(seed)        self.dead_frac = dead_frac        self._precompute(rng)    def _precompute(self, rng):        self.bfs, self.greens, self.reds = [], [], []        for _ in range(self.n):            size = self.size            yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)            bf = rng.uniform(0.25, 0.45, (size, size)).astype(np.float32)            green = np.zeros((size, size), np.float32)            red = np.zeros((size, size), np.float32)            n_cells = int(rng.integers(10, 20))            for _c in range(n_cells):                cx, cy = rng.uniform(15, size - 15, 2)                r = rng.uniform(4, 9)                mask = ((xx - cx) ** 2 + (yy - cy) ** 2) < r ** 2                bf[mask] -= rng.uniform(0.08, 0.18)          # dark nucleus                if rng.random() < self.dead_frac:           # dead: red PI ring                    ring = ((xx - cx) ** 2 + (yy - cy) ** 2)                    ring = (ring < (r + 2) ** 2) & ~mask                    red[ring] = rng.uniform(0.5, 0.95)                else:                                       # viable: green cytosol                    green[mask] = rng.uniform(0.25, 0.6)            # focal blur & sensor noise            from scipy.ndimage import gaussian_filter            bf = gaussian_filter(bf, 1.2)            green = gaussian_filter(green, 1.0)            red = gaussian_filter(red, 0.8)            bf = (bf + rng.normal(0, 0.01, (size, size))).clip(0, 1)            green = (green + rng.normal(0, 0.01, (size, size))).clip(0, 1)            red = (red + rng.normal(0, 0.01, (size, size))).clip(0, 1)            self.bfs.append(bf.astype(np.float32))            self.greens.append(green.astype(np.float32))            self.reds.append(red.astype(np.float32))    def __len__(self) -> int:        return self.n    def __getitem__(self, idx: int):        x = torch.from_numpy(self.bfs[idx])[None].float()        y = torch.from_numpy(np.stack([self.greens[idx], self.reds[idx]], 0)).float()        return x, y, f"synth_{idx}"

In [ ]:
%%writefile src/labelfree/losses.py"""Losses: L1, Fourier-spectrum, perceptual (VGG), and adversarial helpers."""from __future__ import annotationsimport torchimport torch.nn.functional as Fdef l1_loss(pred, target):    return F.l1_loss(pred, target)def fourier_loss(pred, target):    """Spectral loss: L1 on log-magnitude of 2D FFT, keeps high-frequency detail."""    def mag(x):        f = torch.fft.rfft2(x, norm="ortho")        return torch.log1p(torch.abs(f))    return F.l1_loss(mag(pred), mag(target))def perceptual_loss(pred, target, vgg):    """VGG16 relu1_2 + relu2_2 perceptual loss (inputs in [0,1])."""    def feats(x):        x = F.interpolate(x, size=(256, 256), mode="bilinear", align_corners=False)        x = vgg(x)        return x    return F.l1_loss(feats(pred), feats(target))def make_vgg():    """VGG16 features up to relu2_2 (used only if config enables perceptual loss)."""    from torchvision import models    vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features[:9]    for p in vgg.parameters():        p.requires_grad_(False)    return vgg.eval()

In [ ]:
%%writefile src/labelfree/discriminator.py"""PatchGAN-style discriminator for adversarial training (optional)."""from __future__ import annotationsimport torch.nn as nnimport torch.nn.functional as Fclass PatchDiscriminator(nn.Module):    def __init__(self, in_ch: int = 3, base: int = 32):        super().__init__()        self.net = nn.Sequential(            nn.Conv2d(in_ch, base, 4, 2, 1, bias=False),            nn.LeakyReLU(0.2),            nn.Conv2d(base, base * 2, 4, 2, 1, bias=False),            nn.GroupNorm(8, base * 2), nn.LeakyReLU(0.2),            nn.Conv2d(base * 2, base * 4, 4, 2, 1, bias=False),            nn.GroupNorm(8, base * 4), nn.LeakyReLU(0.2),            nn.Conv2d(base * 4, 1, 4, 1, 1),        )    def forward(self, x):        return self.net(x)def hinge_d_loss(d_real, d_fake) -> float:    """Hinge adversarial loss; returns Python float (sum of mean terms)."""    return F.relu(1.0 - d_real).mean() + F.relu(1.0 + d_fake).mean()def hinge_g_loss(d_fake):    return -d_fake.mean()

In [ ]:
%%writefile src/labelfree/uncertainty.py
"""MC-dropout uncertainty estimation for the translator."""
from __future__ import annotations

import torch


@torch.no_grad()
def mc_dropout_infer(model, x, n: int = 3) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (mean_pred, std_pred) over n stochastic forward passes.

    The model must contain dropout layers; dropout is active in eval via
    model.train() so `n` samples approximate the predictive posterior.
    """
    model.train()
    preds = []
    for _ in range(n):
        preds.append(model(x).unsqueeze(0))
        from .utils import mark_step
        mark_step(x.device)
    model.eval()
    stack = torch.cat(preds, 0)          # [n, C, H, W]
    mean = stack.mean(0)
    std = stack.std(0, unbiased=False)
    return mean, std


In [ ]:
%%writefile src/labelfree/segment.py"""Segmentation: label-free cell/dead masks from predicted fluorescence.Adaptive percentile thresholds make the readout robust to the compresseddynamic range of virtual stains (and to any microscope)."""from __future__ import annotationsimport numpy as npfrom scipy import ndimagefrom skimage import measure, morphologydef _threshold(img: np.ndarray, p: float, k: float, floor: float) -> float:    return max(floor, float(np.percentile(img, p)) * k)def dead_mask(red: np.ndarray, p: float = 99.0, k: float = 0.75,              min_size: int = 30, floor: float = 0.05) -> np.ndarray:    """Dead-cell mask: PI-positive nuclei after smoothing + adaptive threshold."""    r = ndimage.gaussian_filter(red, 1.0)    m = r > _threshold(r, p, k, floor)    m = morphology.remove_small_objects(m, max_size=max(1, min_size - 1))    m = ndimage.binary_closing(m, iterations=1)    return mdef viable_mask(green: np.ndarray, p: float = 99.0, k: float = 0.75,                min_size: int = 60, floor: float = 0.05) -> np.ndarray:    g = ndimage.gaussian_filter(green, 1.2)    m = g > _threshold(g, p, k, floor)    m = morphology.remove_small_objects(m, max_size=max(1, min_size - 1))    return mdef cell_stats(viable: np.ndarray, dead: np.ndarray) -> dict:    n_dead = len(np.unique(measure.label(dead))) - 1    n_viable = len(np.unique(measure.label(viable))) - 1    dead_area = float(dead.mean())    viable_area = float(viable.mean())    return {        "n_dead": int(n_dead), "n_viable": int(n_viable),        "dead_area_frac": dead_area, "viable_area_frac": viable_area,        "viability": float(1.0 - min(1.0, n_dead / max(1, n_dead + n_viable))),    }

In [ ]:
%%writefile src/labelfree/phenotype.py"""Phenotype scoring and Hill dose-response fitting."""from __future__ import annotationsimport jsonfrom pathlib import Pathimport numpy as npimport pandas as pdfrom scipy.optimize import curve_fitfrom .segment import cell_stats, dead_mask, viable_maskfrom .utils import read_image_graydef phenotype_field(green: np.ndarray, red: np.ndarray) -> dict:    return cell_stats(viable_mask(green), dead_mask(red))def phenotype_directory(pred_dir: str | Path, out_json: str | Path | None = None,                        ch_green: str = "green", ch_red: str = "red") -> pd.DataFrame:    pred_dir = Path(pred_dir)    rows = []    for p in sorted(pred_dir.glob(f"*_pred_{ch_green}.png")):        sid = p.name.replace(f"_pred_{ch_green}.png", "")        g = read_image_gray(p)        r = read_image_gray(pred_dir / f"{sid}_pred_{ch_red}.png")        s = phenotype_field(g, r)        s["id"] = sid        rows.append(s)    df = pd.DataFrame(rows)    if out_json:        Path(out_json).parent.mkdir(parents=True, exist_ok=True)        Path(out_json).write_text(json.dumps(df.to_dict("records"), indent=2))    return dfdef _hill(x, Emax, EC50, n):    return Emax * x ** n / (EC50 ** n + x ** n)def fit_hill(doses: np.ndarray, responses: np.ndarray):    """Fit Emax*D^n/(EC50^n + D^n); returns dict or None."""    doses = np.asarray(doses, float)    responses = np.asarray(responses, float)    if len(doses) < 4 or np.any(doses <= 0):        return None    try:        p0 = [float(np.max(responses) - np.min(responses)), float(np.median(doses)), 1.0]        popt, _ = curve_fit(_hill, doses, responses, p0=p0, maxfev=20000)        Emax, EC50, n = popt        rss = float(np.sum((_hill(doses, *popt) - responses) ** 2))        tss = float(np.sum((responses - responses.mean()) ** 2))        r2 = 1.0 - rss / tss if tss > 0 else 0.0        return {"EC50": float(EC50), "Emax": float(Emax), "Hill_n": float(n), "R2": float(r2)}    except Exception:        return None

In [ ]:
%%writefile src/labelfree/plot.py"""Plotting helpers: montage grid and metrics bar chart."""from __future__ import annotationsimport numpy as npimport matplotlibmatplotlib.use("Agg")import matplotlib.pyplot as pltdef _crop_square(arr, size=256):    h, w = arr.shape[:2]    top, left = (h - size) // 2, (w - size) // 2    return arr[top:top + size, left:left + size]def montage(inputs, targets, preds, stds=None, n: int = 4, path: str = "montage.png",            crop: int = 256):    """Grid: rows = fields, columns = BF | target green | predicted green | std (if given)."""    n = min(n, len(inputs), len(targets), len(preds))    ncols = 4 if stds else 3    fig, axes = plt.subplots(n, ncols, figsize=(ncols * 3.2, n * 3.2))    if n == 1:        axes = axes[None, :]    labels = ["Bright-field", "Target (green)", "Virtual stain (green)"] + (["Uncertainty"] if stds else [])    for i in range(n):        inp = _crop_square(inputs[i], crop)        tgt = _crop_square(targets[i], crop)        prd = _crop_square(preds[i], crop)        imgs = [inp, tgt, prd]        if stds:            imgs.append(_crop_square(stds[i], crop))        for j, img in enumerate(imgs):            ax = axes[i, j]            ax.imshow(img, cmap="gray")            ax.set_xticks([]); ax.set_yticks([])            if i == 0:                ax.set_title(labels[j], fontsize=11)    fig.tight_layout()    fig.savefig(path, dpi=150, bbox_inches="tight")    plt.close(fig)def metrics_bar(summary: dict, path: str = "metrics.png"):    ch = summary.get("channels", {})    names = list(ch.keys())    psnr = [ch[c]["psnr"] for c in names]    ssim = [ch[c]["ssim"] for c in names]    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))    axes[0].bar(names, psnr, color="#4C72B0")    axes[0].set_ylabel("PSNR (dB)")    axes[0].set_ylim(0, max(psnr) * 1.25)    for x, v in zip(names, psnr):        axes[0].text(x, v + 0.3, f"{v:.2f}", ha="center", fontsize=10)    axes[1].bar(names, ssim, color="#DD8452")    axes[1].set_ylabel("SSIM")    axes[1].set_ylim(0, 1.1)    for x, v in zip(names, ssim):        axes[1].text(x, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)    fig.tight_layout()    fig.savefig(path, dpi=150, bbox_inches="tight")    plt.close(fig)

In [ ]:
%%writefile src/labelfree/evaluate.py
"""Evaluation: PSNR / SSIM over a loader (used during training and final eval)."""
from __future__ import annotations

import numpy as np
import torch
import torch.nn.functional as F
from skimage.metrics import peak_signal_noise_ratio, structural_similarity


def _to_np(t):
    return t.detach().cpu().numpy()


@torch.no_grad()
def eval_metrics(model, loader, device) -> tuple[float, float]:
    """Pooled PSNR / SSIM across channels and samples."""
    model.eval()
    psnrs, ssims = [], []
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        p = _to_np(pred)
        t = _to_np(y)
        for c in range(p.shape[1]):
            p_c = np.clip(p[0, c], 0, 1)
            t_c = np.clip(t[0, c], 0, 1)
            psnrs.append(peak_signal_noise_ratio(t_c, p_c, data_range=1.0))
            ssims.append(structural_similarity(t_c, p_c, data_range=1.0))
    model.train()
    return float(np.mean(psnrs)), float(np.mean(ssims))


@torch.no_grad()
def eval_per_channel(model, loader, device, ch_names=("green", "red")):
    """Per-channel PSNR/SSIM lists."""
    model.eval()
    acc = {c: {"psnr": [], "ssim": []} for c in ch_names}
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        p = _to_np(pred)
        t = _to_np(y)
        for ci, c in enumerate(ch_names):
            p_c = np.clip(p[0, ci], 0, 1)
            t_c = np.clip(t[0, ci], 0, 1)
            acc[c]["psnr"].append(peak_signal_noise_ratio(t_c, p_c, data_range=1.0))
            acc[c]["ssim"].append(structural_similarity(t_c, p_c, data_range=1.0))
    model.train()
    return {c: {k: float(np.mean(v)) for k, v in d.items()} for c, d in acc.items()}


In [ ]:
%%writefile src/labelfree/train.py
"""Training CLI: `python -m labelfree.train --config configs/train_*.yaml`."""
from __future__ import annotations

import argparse
import csv
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from .datasets import Caco2PairsDataset, SyntheticPairsDataset
from .losses import fourier_loss, l1_loss, make_vgg, perceptual_loss
from .models import AttResUNet
from .utils import get_device, load_config, mark_step, set_seed
from .evaluate import eval_metrics


def build_datasets(cfg):
    if cfg["data"].get("synthetic", False):
        n = cfg["data"].get("n", 64)
        train = SyntheticPairsDataset(n=n, size=cfg["data"]["crop"], seed=0)
        val = SyntheticPairsDataset(n=max(8, n // 8), size=cfg["data"]["crop"], seed=1)
        return train, val
    d = cfg["data"]
    train = Caco2PairsDataset(d["pairs_csv"], d["root"], split="train", crop=d["crop"])
    val = Caco2PairsDataset(d["pairs_csv"], d["root"], split="val", crop=d["crop"])
    if d.get("limit"):
        train = torch.utils.data.Subset(train, list(range(min(d["limit"], len(train)))))
    return train, val


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--out", default=None)
    args = ap.parse_args()

    cfg = load_config(args.config)
    set_seed(cfg.get("seed", 42))
    device = get_device()
    out = Path(args.out or cfg["out"])
    out.mkdir(parents=True, exist_ok=True)

    m = cfg["model"]
    model = AttResUNet(in_ch=cfg["data"]["in_ch"], out_ch=cfg["data"]["out_ch"],
                       base=m["base"], depth=m["depth"], attn=m.get("attn", True),
                       dropout=float(m.get("dropout", 0.1))).to(device)
    print(f"model params={sum(p.numel() for p in model.parameters())/1e6:.2f}M device={device}")

    opt = torch.optim.AdamW(model.parameters(), lr=float(cfg["train"]["lr"]),
                            weight_decay=float(cfg["train"].get("wd", 0.0)))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=cfg["train"]["epochs"])

    train_ds, val_ds = build_datasets(cfg)
    train_loader = DataLoader(train_ds, batch_size=cfg["train"]["batch"],
                              shuffle=True, num_workers=0, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

    vgg = make_vgg().to(device) if cfg["train"].get("perceptual") else None
    w_f = float(cfg["train"].get("w_fourier", 0.0))
    w_p = float(cfg["train"].get("w_perceptual", 0.0))

    log_path = out / "train_log.csv"
    best_psnr = -1.0
    with open(log_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "time_s", "loss_total", "loss_l1", "val_psnr", "val_ssim"])

        for epoch in range(1, cfg["train"]["epochs"] + 1):
            t0 = time.time()
            model.train()
            total_loss, total_l1, n_batch = 0.0, 0.0, 0
            for x, y, _ in train_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                loss = l1_loss(pred, y)
                if w_f > 0:
                    loss = loss + w_f * fourier_loss(pred, y)
                if w_p > 0:
                    loss = loss + w_p * perceptual_loss(pred, y, vgg)
                opt.zero_grad()
                loss.backward()
                opt.step()
                mark_step(device)
                total_loss += loss.item()
                total_l1 += l1_loss(pred, y).item()
                n_batch += 1
            sched.step()

            val_psnr, val_ssim = eval_metrics(model, val_loader, device)
            mark_step(device)
            elapsed = time.time() - t0
            print(f"epoch {epoch:02d} loss={total_loss/n_batch:.4f} l1={total_l1/n_batch:.4f} "
                  f"val_psnr={val_psnr:.3f} val_ssim={val_ssim:.4f} ({elapsed:.0f}s)")
            writer.writerow([epoch, round(elapsed, 1), round(total_loss / n_batch, 5),
                             round(total_l1 / n_batch, 5), round(val_psnr, 4), round(val_ssim, 4)])
            f.flush()
            if val_psnr > best_psnr:
                best_psnr = val_psnr
                torch.save({"state_dict": model.state_dict(), "epoch": epoch,
                            "val_psnr": val_psnr, "val_ssim": val_ssim},
                           out / "best.ckpt")
    print(f"done -> {out} (best val PSNR {best_psnr:.3f})")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/labelfree/infer.py"""Inference CLI: `python -m labelfree.infer --config configs/infer.yaml ...`."""from __future__ import annotationsimport argparseimport loggingfrom pathlib import Pathimport torchfrom torch.utils.data import DataLoaderfrom .datasets import Caco2PairsDatasetfrom .models import AttResUNetfrom .uncertainty import mc_dropout_inferfrom .utils import get_device, load_config, save_tensor_image, set_seedlog = logging.getLogger("infer")logging.basicConfig(level=logging.INFO, format="[%(asctime)s %(levelname)s] %(message)s")def main():    ap = argparse.ArgumentParser()    ap.add_argument("--config", required=True)    ap.add_argument("--ckpt", required=True)    ap.add_argument("--csv", required=True)    ap.add_argument("--root", required=True)    ap.add_argument("--split", default="test")    ap.add_argument("--crop", type=int, default=256)    ap.add_argument("--mc-samples", type=int, default=3)    ap.add_argument("--out", required=True)    ap.add_argument("--limit", type=int, default=0)    ap.add_argument("--base", type=int, default=None, help="override model base width")    ap.add_argument("--depth", type=int, default=None, help="override model depth")    args = ap.parse_args()    set_seed(42)    device = get_device()    cfg = load_config(args.config)    m = cfg["model"]    model = AttResUNet(in_ch=cfg["data"]["in_ch"], out_ch=cfg["data"]["out_ch"],                       base=args.base if args.base else m["base"],                       depth=args.depth if args.depth else m["depth"],                       attn=m.get("attn", True),                       dropout=float(m.get("dropout", 0.1))).to(device)    ckpt = torch.load(args.ckpt, map_location=device, weights_only=False)    model.load_state_dict(ckpt["state_dict"])    model.eval()    log.info("loaded %s (epoch=%s)", args.ckpt, ckpt.get("epoch"))    ds = Caco2PairsDataset(args.csv, args.root, split=args.split, crop=args.crop)    if args.limit:        ds.df = ds.df.iloc[: args.limit]    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)    out = Path(args.out)    out.mkdir(parents=True, exist_ok=True)    n = 0    with torch.no_grad():        for x, y, sid in loader:            if isinstance(sid, (tuple, list)):                sid = sid[0]            x = x.to(device)            pred, std = mc_dropout_infer(model, x, n=args.mc_samples)            save_tensor_image(x[0].cpu(), out / f"{sid}_input.png")            save_tensor_image(pred[0, 0:1].cpu(), out / f"{sid}_pred_green.png")            save_tensor_image(pred[0, 1:2].cpu(), out / f"{sid}_pred_red.png")            save_tensor_image(std[0, 0:1].cpu(), out / f"{sid}_std_green.png")            save_tensor_image(std[0, 1:2].cpu(), out / f"{sid}_std_red.png")            n += 1            log.info("inferred %d/%d  %s", n, len(loader), sid)    print(f"done: {n} images -> {out}")if __name__ == "__main__":    main()

In [ ]:
%%writefile src/labelfree/evaluate_cli.py"""Evaluation CLI over saved predictions:`python -m labelfree.evaluate --csv data/caco2/pairs.csv --root data/caco2    --pred-dir runs/xxx/infer --split test --crop 256 --out runs/xxx/eval`Writes summary.json (overall + per-channel PSNR/SSIM)."""from __future__ import annotationsimport argparseimport jsonfrom pathlib import Pathimport numpy as npimport pandas as pdfrom skimage.metrics import peak_signal_noise_ratio, structural_similarityfrom .utils import center_crop_np, read_image_graydef main():    ap = argparse.ArgumentParser()    ap.add_argument("--csv", required=True)    ap.add_argument("--root", required=True)    ap.add_argument("--pred-dir", required=True)    ap.add_argument("--split", default="test")    ap.add_argument("--crop", type=int, default=256)    ap.add_argument("--out", required=True)    args = ap.parse_args()    df = pd.read_csv(args.csv)    if args.split != "all":        df = df[df["split"] == args.split]    pred_dir = Path(args.pred_dir)    acc = {"green": {"psnr": [], "ssim": []}, "red": {"psnr": [], "ssim": []}}    n = 0    for _, row in df.iterrows():        sid = row["id"]        target = read_image_gray(f"{args.root}/{row['green']}")        if args.crop:            target = center_crop_np(target, args.crop)        for ch, col in (("green", row["green"]), ("red", row["red"])):            pred_p = pred_dir / f"{sid}_pred_{ch}.png"            if not pred_p.exists():                continue            pred = read_image_gray(pred_p)            tgt = center_crop_np(read_image_gray(f"{args.root}/{col}"), args.crop) if args.crop \                else read_image_gray(f"{args.root}/{col}")            acc[ch]["psnr"].append(peak_signal_noise_ratio(tgt, pred, data_range=1.0))            acc[ch]["ssim"].append(structural_similarity(tgt, pred, data_range=1.0))        n += 1    summary = {"n_samples": n, "split": args.split, "crop": args.crop, "channels": {}}    all_psnr, all_ssim = [], []    for ch, d in acc.items():        if d["psnr"]:            summary["channels"][ch] = {"psnr": float(np.mean(d["psnr"])),                                       "ssim": float(np.mean(d["ssim"]))}            all_psnr += d["psnr"]            all_ssim += d["ssim"]    if all_psnr:        summary["overall"] = {"psnr": float(np.mean(all_psnr)), "ssim": float(np.mean(all_ssim))}    out = Path(args.out)    out.mkdir(parents=True, exist_ok=True)    (out / "summary.json").write_text(json.dumps(summary, indent=2))    print(json.dumps(summary, indent=2))    print(f"wrote {out / 'summary.json'}")if __name__ == "__main__":    main()

In [ ]:
%%writefile src/labelfree/agent.py"""Minimal AI-agent demo: conversational driver over the pipeline.Usage (after training):    python -m labelfree.agent --ckpt runs/caco2_mini/best.ckpt --root data/caco2Then type commands: `phenotype <field_id>`, `hill`, `montage`, `exit`."""from __future__ import annotationsimport argparseimport jsonfrom .datasets import Caco2PairsDatasetfrom .uncertainty import mc_dropout_inferfrom .models import AttResUNetfrom .phenotype import fit_hill, phenotype_fieldfrom .plot import montagefrom .utils import get_device, load_config, read_image_gray, save_tensor_image, set_seedimport torchdef main():    ap = argparse.ArgumentParser()    ap.add_argument("--config", default="configs/infer.yaml")    ap.add_argument("--ckpt", required=True)    ap.add_argument("--root", default="data/caco2")    ap.add_argument("--csv", default="data/caco2/pairs.csv")    ap.add_argument("--crop", type=int, default=256)    args = ap.parse_args()    set_seed(42)    device = get_device()    cfg = load_config(args.config)    m = cfg["model"]    model = AttResUNet(in_ch=cfg["data"]["in_ch"], out_ch=cfg["data"]["out_ch"],                       base=m["base"], depth=m["depth"], attn=m.get("attn", True),                       dropout=float(m.get("dropout", 0.1))).to(device)    model.load_state_dict(torch.load(args.ckpt, map_location=device, weights_only=False)["state_dict"])    model.eval()    print("labelfree agent ready. commands: phenotype <id> | montage [n] | hill <doses.csv> | exit")    while True:        try:            cmd = input(">> ").strip().split()        except EOFError:            break        if not cmd:            continue        if cmd[0] == "exit":            break        if cmd[0] == "phenotype" and len(cmd) > 1:            sid = cmd[1]            bf = read_image_gray(f"{args.root}/{sid}")            import numpy as np            from .utils import center_crop_np            x = torch.from_numpy(center_crop_np(bf, args.crop))[None, None].to(device)            pred, _ = mc_dropout_infer(model, x, n=3)            g = pred[0, 0].clamp(0, 1).cpu().numpy()            r = pred[0, 1].clamp(0, 1).cpu().numpy()            stats = phenotype_field(g, r)            print(json.dumps(stats, indent=2))        elif cmd[0] == "montage":            save_tensor_image(torch.zeros(1), "/tmp/labelfree_agent_placeholder.png")            print("run scripts/make_figures.py to render the montage")        elif cmd[0] == "hill" and len(cmd) > 1:            import pandas as pd            d = pd.read_csv(cmd[1])            fit = fit_hill(d["dose"].values, d["response"].values)            print(json.dumps(fit, indent=2) if fit else "fit failed")        else:            print("unknown command")if __name__ == "__main__":    main()

In [ ]:
%%writefile src/labelfree/models/__init__.pyfrom .unet import AttResUNet__all__ = ["AttResUNet"]

In [ ]:
%%writefile src/labelfree/models/unet.py"""AttResUNet: attention-gated residual U-Net for image-to-image translation."""from __future__ import annotationsimport torchimport torch.nn as nnimport torch.nn.functional as Fclass ConvBlock(nn.Module):    """Two conv + group-norm + ReLU with a residual shortcut and MC-dropout."""    def __init__(self, cin: int, cout: int, norm_groups: int = 8, dropout: float = 0.1):        super().__init__()        self.conv1 = nn.Conv2d(cin, cout, 3, padding=1, bias=False)        self.n1 = nn.GroupNorm(norm_groups, cout)        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1, bias=False)        self.n2 = nn.GroupNorm(norm_groups, cout)        self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()        self.shortcut = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()    def forward(self, x):        h = F.relu(self.n1(self.conv1(x)))        h = F.relu(self.n2(self.conv2(h)))        h = self.drop(h)        return h + self.shortcut(x)class SelfAttn(nn.Module):    """Spatial self-attention (query/key/value 1x1 convs) at deepest level."""    def __init__(self, c: int):        super().__init__()        self.q = nn.Conv2d(c, c // 4, 1)        self.k = nn.Conv2d(c, c // 4, 1)        self.v = nn.Conv2d(c, c, 1)        self.gamma = nn.Parameter(torch.zeros(1))    def forward(self, x):        b, c, h, w = x.shape        q = self.q(x).flatten(2).transpose(1, 2)     # [b, hw, c/4]        k = self.k(x).flatten(2)                     # [b, c/4, hw]        v = self.v(x).flatten(2).transpose(1, 2)     # [b, hw, c]        attn = torch.softmax(q @ k / (c ** 0.5), dim=-1)        out = (attn @ v).transpose(1, 2).reshape(b, c, h, w)        return self.gamma * out + xclass AttResUNet(nn.Module):    def __init__(self, in_ch: int = 1, out_ch: int = 2, base: int = 32,                 depth: int = 3, attn: bool = True, norm_groups: int = 8,                 dropout: float = 0.1):        super().__init__()        self.depth = depth        self.enc = nn.ModuleList()        self.downs = nn.ModuleList()        cin = in_ch        for d in range(depth):            cout = base * (2 ** d)            self.enc.append(ConvBlock(cin, cout, norm_groups, dropout))            if d < depth - 1:                self.downs.append(nn.MaxPool2d(2))            cin = cout        self.attn = SelfAttn(cin) if attn else nn.Identity()        self.dec = nn.ModuleList()        self.ups = nn.ModuleList()        for d in range(depth - 2, -1, -1):            cout = base * (2 ** d)            self.ups.append(nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False))            self.dec.append(ConvBlock(cin + cout, cout, norm_groups, dropout))            cin = cout        self.head = nn.Conv2d(cin, out_ch, 1)    def forward(self, x):        skips = []        h = x        for d, blk in enumerate(self.enc):            h = blk(h)            if d < self.depth - 1:                skips.append(h)                h = self.downs[d](h)        h = self.attn(h)        for up, blk, sk in zip(self.ups, self.dec, reversed(skips)):            h = up(h)            h = torch.cat([h, sk], dim=1)            h = blk(h)        return self.head(h)

In [ ]:
%%writefile configs/train_caco2.yaml# Full real-data run — GPU (base64 / depth4 / 256 crop / 60 epochs / all 178 train fields)# Run inside a Kaggle GPU notebook (see notebooks/kaggle_train.py).seed: 42out: runs/caco2data:  synthetic: false  pairs_csv: data/caco2/pairs.csv  root: data/caco2/raw  crop: 256  limit: 0  in_ch: 1  out_ch: 2model:  base: 64  depth: 4  attn: true  dropout: 0.1train:  epochs: 60  batch: 16  lr: 0.0004  wd: 0.0001  w_fourier: 0.5  w_perceptual: 0.05

In [ ]:
%%writefile configs/train_caco2_mini.yaml# Mini real-data run — CPU-friendly (base32 / depth3 / 192 crop / 8 epochs / 120 fields)seed: 42out: runs/caco2_minidata:  synthetic: false  pairs_csv: data/caco2/pairs.csv  root: data/caco2/raw  crop: 192  limit: 120  in_ch: 1  out_ch: 2model:  base: 32  depth: 3  attn: true  dropout: 0.1train:  epochs: 30  batch: 8  lr: 0.0006  wd: 0.0001  w_fourier: 0.5  w_perceptual: 0.0

In [ ]:
%%writefile configs/infer.yaml# Inference configuration (shared by infer / evaluate CLIs)seed: 42data:  in_ch: 1  out_ch: 2model:  base: 32  depth: 3  attn: true  dropout: 0.1

In [ ]:
%%writefile scripts/make_figures.py"""Generate report figures from a finished run.Usage:  python scripts/make_figures.py --run runs/caco2_mini --out report/assetsProduces <out>/caco2_montage.png and <out>/metrics.png."""from __future__ import annotationsimport argparseimport jsonimport sysfrom pathlib import Pathimport pandas as pdROOT = Path(__file__).resolve().parents[1]sys.path.insert(0, str(ROOT / "src"))from labelfree.plot import metrics_bar, montage           # noqa: E402from labelfree.utils import read_image_gray               # noqa: E402def main():    ap = argparse.ArgumentParser()    ap.add_argument("--run", default="runs/caco2_mini")    ap.add_argument("--out", default="report/assets")    ap.add_argument("--root", default="data/caco2/raw")    ap.add_argument("--n", type=int, default=4)    args = ap.parse_args()    run = Path(args.run)    out = Path(args.out)    out.mkdir(parents=True, exist_ok=True)    pred_dir = run / "infer"    eval_sum = json.load(open(run / "eval" / "summary.json"))    pairs = pd.read_csv(ROOT / "data" / "caco2" / "pairs.csv")    path_of = dict(zip(pairs["id"], pairs["green"]))    preds = sorted(pred_dir.glob("*_pred_green.png"))[: args.n]    inputs, targets, pred_imgs, stds = [], [], [], []    for p in preds:        sid = p.name.replace("_pred_green.png", "")        inputs.append(read_image_gray(pred_dir / f"{sid}_input.png"))        targets.append(read_image_gray(ROOT / "data" / "caco2" / "raw" / path_of[sid]))        pred_imgs.append(read_image_gray(p))        s = pred_dir / f"{sid}_std_green.png"        stds.append(read_image_gray(s) if s.exists() else None)    has_std = all(s is not None for s in stds)    montage(inputs, targets, pred_imgs, stds if has_std else None,            n=len(inputs), path=str(out / "caco2_montage.png"))    print("montage ->", out / "caco2_montage.png")    metrics_bar(eval_sum, path=str(out / "metrics.png"))    print("metrics  ->", out / "metrics.png")if __name__ == "__main__":    main()

In [ ]:
%%writefile scripts/viability_agreement.py"""Viability agreement: predicted fluorescence -> viability vs real stained viability.Usage:  python scripts/viability_agreement.py --run runs/caco2_miniWrites runs/<name>/viability_agreement.json + figure to report/assets/viability_agreement.png"""from __future__ import annotationsimport argparseimport jsonimport sysfrom pathlib import Pathimport numpy as npimport pandas as pdROOT = Path(__file__).resolve().parents[1]sys.path.insert(0, str(ROOT / "src"))from labelfree.phenotype import phenotype_field          # noqa: E402from labelfree.utils import read_image_gray              # noqa: E402def viability_of(g, r):    return phenotype_field(g, r)["viability"]def main():    ap = argparse.ArgumentParser()    ap.add_argument("--run", default="runs/caco2_mini")    ap.add_argument("--csv", default="data/caco2/pairs.csv")    ap.add_argument("--root", default="data/caco2/raw")    ap.add_argument("--split", default="test")    ap.add_argument("--out", default=str(ROOT / "report" / "assets"),                    help="output dir for the agreement figure")    args = ap.parse_args()    run = Path(args.run)    df = pd.read_csv(args.csv)    df = df[df["split"] == args.split]    pred_dir = run / "infer"    true_viab, pred_viab, ids = [], [], []    for _, row in df.iterrows():        sid = row["id"]        p = pred_dir / f"{sid}_pred_green.png"        if not p.exists():            continue        pred_viab.append(viability_of(            read_image_gray(p), read_image_gray(pred_dir / f"{sid}_pred_red.png")))        true_viab.append(viability_of(            read_image_gray(f"{args.root}/{row['green']}"),            read_image_gray(f"{args.root}/{row['red']}")))        ids.append(sid)    t = np.array(true_viab); v = np.array(pred_viab)    pearson = float(np.corrcoef(t, v)[0, 1])    from scipy.stats import spearmanr    spearman = float(spearmanr(t, v).statistic)    mae = float(np.mean(np.abs(t - v)))    result = {"n": int(len(t)), "pearson": pearson, "spearman": spearman, "mae": mae}    print(json.dumps(result, indent=2))    (run / "viability_agreement.json").write_text(json.dumps(result, indent=2))    import matplotlib    matplotlib.use("Agg")    import matplotlib.pyplot as plt    fig, ax = plt.subplots(figsize=(5.5, 5))    ax.scatter(t, v, s=40, alpha=0.7, edgecolor="k", linewidth=0.5)    lims = [min(t.min(), v.min()) - 0.05, max(t.max(), v.max()) + 0.05]    ax.plot(lims, lims, "--", color="gray", label="y=x")    ax.set_xlabel("viability from real stain")    ax.set_ylabel("viability from virtual stain")    ax.set_title(f"Pearson r={pearson:.3f}  Spearman ρ={spearman:.3f}  MAE={mae:.3f}")    ax.legend()    ax.set_xlim(lims); ax.set_ylim(lims)    plt.tight_layout()    out_png = Path(args.out) / "viability_agreement.png"    out_png.parent.mkdir(parents=True, exist_ok=True)    fig.savefig(out_png, dpi=150)    plt.close(fig)    print("saved", out_png)if __name__ == "__main__":    main()

In [ ]:
%%writefile scripts/rxrx3_dose_response.py"""Module C — RxRx3-core dose-response phenomics (CPU, ~5 min with data cached).Downloads metadata + OpenPhenom embeddings, computes per-well within-plateperturbation scores, aggregates per (compound, dose), fits Hill curves, andwrites runs/rxrx3/{dose_response_curves.csv, dose_response_fits.json, control_sanity.png}.Usage:  python scripts/rxrx3_dose_response.py --out runs/rxrx3"""from __future__ import annotationsimport argparseimport jsonimport sysfrom pathlib import Pathimport numpy as npimport pandas as pdROOT = Path(__file__).resolve().parents[1]sys.path.insert(0, str(ROOT / "src"))HF_REPO = "recursionpharma/rxrx3-core"META_FILE = "metadata_rxrx3_core.csv"EMB_FILE = "OpenPhenom_rxrx3_core_embeddings.parquet"def load_hf(filename: str) -> Path:    from huggingface_hub import hf_hub_download    return Path(hf_hub_download(HF_REPO, filename, repo_type="dataset"))def main():    ap = argparse.ArgumentParser()    ap.add_argument("--out", default="runs/rxrx3")    args = ap.parse_args()    out = Path(args.out)    out.mkdir(parents=True, exist_ok=True)    print("[1/4] loading metadata + embeddings ...")    meta_path = load_hf(META_FILE)    emb_path = load_hf(EMB_FILE)    meta = pd.read_csv(meta_path)    emb = pd.read_parquet(emb_path)    print(f"      metadata: {len(meta)} rows | embeddings: {len(emb)} rows")    id_col = "well_id" if "well_id" in meta.columns else meta.columns[0]    emb_id_col = "well_id" if "well_id" in emb.columns else emb.columns[0]    df = meta.merge(emb, left_on=id_col, right_on=emb_id_col, how="inner", suffixes=("", "_e"))    # keep only compound wells (drop CRISPR)    if "perturbation_type" in df.columns:        df = df[df["perturbation_type"] == "COMPOUND"].reset_index(drop=True)    feat_cols = [c for c in df.columns if c.startswith("feature_") or c.startswith("feat_")                 or c.startswith("emb_")]    if not feat_cols:        feat_cols = [c for c in df.columns                     if df[c].dtype in (np.float32, np.float64) and c != id_col]    print(f"      wells after COMPOUND filter: {len(df)}; features used: {len(feat_cols)}")    compound_col = next((c for c in ("treatment", "compound", "compound_id", "inchi_key") if c in df.columns), None)    dose_col = next((c for c in ("concentration", "dose", "dose_uM") if c in df.columns), None)    plate_col = next((c for c in ("plate", "plate_id", "experiment_plate") if c in df.columns), None)    ctrl_col = "well_type_label" if "well_type_label" in df.columns else None    print(f"      columns: compound={compound_col} dose={dose_col} plate={plate_col} ctrl={ctrl_col}")    if not (compound_col and dose_col and plate_col):        raise SystemExit("column layout not recognized — inspect df.columns and adapt")    F = df[feat_cols].to_numpy(np.float32)    print("[2/4] within-plate control-normalized perturbation scores ...")    ctrl_mask = df[compound_col].astype(str).str.lower().str.contains(        "control|dmso|empty", na=False)    rows = []    for plate, g in df.groupby(plate_col):        cm = ctrl_mask.loc[g.index]        if cm.sum() < 5:            cm = pd.Series([False] * len(g))        centroid = F[g.index[cm.values]].mean(0) if cm.any() else F[g.index].mean(0)        scores = np.linalg.norm(F[g.index] - centroid, axis=1)        for i, idx in enumerate(g.index):            rows.append({"plate": int(plate), "well": df.loc[idx, id_col],                         "compound": df.loc[idx, compound_col],                         "dose": float(df.loc[idx, dose_col]), "score": float(scores[i])})    scores = pd.DataFrame(rows)    print("[3/4] sanity check: negative controls near zero ...")    ctrl_scores = scores[scores["compound"].astype(str).str.lower().str.contains(        "control|dmso|empty", na=False)]    treat_scores = scores[~scores["compound"].astype(str).str.lower().str.contains(        "control|dmso|empty", na=False)]    print(f"      control score mean={ctrl_scores['score'].mean():.4f}  "          f"treated score mean={treat_scores['score'].mean():.4f}")    assert ctrl_scores["score"].mean() < treat_scores["score"].mean(), "sanity check failed"    print("[4/4] dose-response aggregation + Hill fit ...")    from labelfree.phenotype import fit_hill    agg = scores.groupby(["compound", "dose"])["score"].median().reset_index()    fits = {}    for compound, g in agg.groupby("compound"):        if len(g) < 4:            continue        fit = fit_hill(g["dose"].values, g["score"].values)        if fit:            fits[compound] = {**fit, "n_doses": int(len(g)), "max_score": float(g["score"].max())}    fits_sorted = sorted(fits.items(), key=lambda kv: kv[1]["max_score"], reverse=True)    print(f"      fitted compounds: {len(fits_sorted)}")    for name, f in fits_sorted[:5]:        print(f"        {name}: EC50={f['EC50']:.3f} Emax={f['Emax']:.3f} R2={f['R2']:.3f}")    agg.to_csv(out / "dose_response_curves.csv", index=False)    with open(out / "dose_response_fits.json", "w") as f:        json.dump({k: v for k, v in fits_sorted}, f, indent=2)    import matplotlib    matplotlib.use("Agg")    import matplotlib.pyplot as plt    fig, ax = plt.subplots(figsize=(6, 4))    ax.hist(ctrl_scores["score"], bins=40, alpha=0.6,            label=f"controls (n={len(ctrl_scores)}, μ={ctrl_scores['score'].mean():.3f})")    ax.hist(treat_scores["score"], bins=60, alpha=0.6,            label=f"treatments (n={len(treat_scores)}, μ={treat_scores['score'].mean():.3f})")    ax.set_xlabel("within-plate perturbation score")    ax.set_ylabel("wells")    ax.legend()    plt.tight_layout()    plt.savefig(out / "control_sanity.png", dpi=150)    plt.close(fig)    # example Hill curve for the strongest hit    if fits_sorted:        top = fits_sorted[0][0]        g = agg[agg["compound"] == top].sort_values("dose")        f = fits_sorted[0][1]        xx = np.logspace(np.log10(g["dose"].min()), np.log10(g["dose"].max()), 100)        yy = f["Emax"] * xx ** f["Hill_n"] / (f["EC50"] ** f["Hill_n"] + xx ** f["Hill_n"])        fig, ax = plt.subplots(figsize=(6, 4))        ax.plot(g["dose"], g["score"], "o", label=top)        ax.plot(xx, yy, "-", label=f"Hill fit: EC50={f['EC50']:.3f} Emax={f['Emax']:.3f} R2={f['R2']:.3f}")        ax.set_xscale("log"); ax.set_xlabel("dose (μM)")        ax.set_ylabel("perturbation score"); ax.legend()        plt.tight_layout()        plt.savefig(out / "hill_example.png", dpi=150)        plt.close(fig)    print(f"done -> {out}")if __name__ == "__main__":    main()

In [ ]:
%%writefile data/download_caco2_fast.py"""Robust parallel downloader with per-chunk retry + zip integrity loop.Usage:  python data/download_caco2_fast.py --out data/caco2/raw --threads 6 --max-attempts 3"""from __future__ import annotationsimport argparseimport threadingimport timeimport zipfilefrom concurrent.futures import ThreadPoolExecutor, as_completedfrom pathlib import Pathfrom urllib.error import HTTPError, URLErrorfrom urllib.request import Request, urlopenFILE_URL = "https://ndownloader.figshare.com/files/38982755"CHUNK = 4 * 1024 * 1024  # 4MB per chunkdef _probe_size(url: str) -> int:    for attempt in range(5):        try:            req = Request(url, method="HEAD")            with urlopen(req, timeout=60) as r:                return int(r.headers["Content-Length"])        except Exception:            time.sleep(2 * (attempt + 1))    raise RuntimeError("cannot probe file size")def _fetch(url: str, start: int, end: int, fp, lock: threading.Lock, progress: list,           nretry: int):    data = None    for attempt in range(nretry):        try:            req = Request(url, headers={"Range": f"bytes={start}-{end}"})            with urlopen(req, timeout=120) as r:                data = r.read()            if len(data) != (end - start + 1):                raise IOError(f"short read {len(data)} != {end - start + 1}")            break        except Exception as e:            if attempt == nretry - 1:                raise            time.sleep(2 + attempt * 3)    with lock:        fp.seek(start)        fp.write(data)        progress[0] += len(data)        print(f"\r{progress[0]/1048576:.1f}/{total_mb:.1f} MB", end="", flush=True)total_mb = 0.0  # set in maindef _download(zip_path: Path, threads: int, nretry: int) -> None:    global total_mb    total = _probe_size(FILE_URL)    total_mb = total / 1048576    print(f"file size: {total_mb:.1f} MB, threads={threads}")    with open(zip_path, "wb") as fp:        fp.truncate(total)        lock = threading.Lock()        progress = [0]        ranges = [(i, min(i + CHUNK - 1, total - 1)) for i in range(0, total, CHUNK)]        with ThreadPoolExecutor(max_workers=threads) as ex:            futs = [ex.submit(_fetch, FILE_URL, s, e, fp, lock, progress, nretry)                    for s, e in ranges]            for f in as_completed(futs):                f.result()   # raises if a chunk failed after retries    print("\ndownload complete")def main():    ap = argparse.ArgumentParser()    ap.add_argument("--out", default="data/caco2/raw")    ap.add_argument("--threads", type=int, default=6)    ap.add_argument("--max-attempts", type=int, default=4,                    help="whole-file attempts (integrity loop)")    ap.add_argument("--keep-zip", action="store_true")    args = ap.parse_args()    out = Path(args.out)    out.mkdir(parents=True, exist_ok=True)    zip_path = out.parent / "2022-09-06_Dataset_Update.zip"    for attempt in range(args.max_attempts):        print(f"=== attempt {attempt + 1}/{args.max_attempts} ===")        try:            _download(zip_path, args.threads, 5)            with zipfile.ZipFile(zip_path) as z:                bad = z.testzip()                if bad:                    raise IOError(f"bad member {bad}")                z.extractall(out)            print("extraction OK")            if not args.keep_zip:                zip_path.unlink()            dirs = sorted([p.name for p in out.iterdir() if p.is_dir()])            n_jpg = sum(len(list((out / d).glob("*.jpg"))) for d in out.iterdir() if d.is_dir())            print(f"{len(dirs)} series folders, jpg total: {n_jpg}")            return        except Exception as e:            print(f"attempt failed: {e}")            time.sleep(3)    raise SystemExit("download failed after all attempts")if __name__ == "__main__":    main()

In [ ]:
%%writefile data/make_caco2_pairs.py"""Build paired field table (pairs.csv) from the raw Caco-2 dataset.Each field has 3 grayscale JPEGs (BF / green / red). Channel roles areauto-detected from mean image intensity: green (calcein) is mid-bright,red (PI nuclei) is darkest on average in these data; the brightest channelis bright-field. If detection disagrees with the folder/camera name, set`--channels bf,green,red` explicitly (suffixes are parsed from filenames).Usage:  python data/make_caco2_pairs.py --root data/caco2/raw --out data/caco2/pairs.csv"""from __future__ import annotationsimport argparseimport refrom pathlib import Pathimport numpy as npimport pandas as pdfrom PIL import Imagedef _mean_gray(p: Path) -> float:    with Image.open(p) as im:        a = np.asarray(im.convert("L"), dtype=np.float32)    return float(a.mean())def main():    ap = argparse.ArgumentParser()    ap.add_argument("--root", default="data/caco2/raw")    ap.add_argument("--out", default="data/caco2/pairs.csv")    ap.add_argument("--channels", default=None,                    help="explicit mapping e.g. 'bf,green,red' from suffix group order")    args = ap.parse_args()    root = Path(args.root)    rows = []    for series in sorted(root.iterdir()):        if not series.is_dir():            continue        # group files by field index; suffixes look like "<k>.0.jpg|<k>.1.jpg|..."        files = sorted(series.glob("*.jpg"))        groups: dict[int, list[tuple[int, Path]]] = {}        for p in files:            m = re.match(r"(\d+)\.(\d+)\.jpg$", p.name)            if not m:                continue            k, s = int(m.group(1)), int(m.group(2))            groups.setdefault(k, []).append((s, p))        for k in sorted(groups):            suff = sorted(groups[k])                      # by suffix            if len(suff) < 3:                continue            sid = f"{series.name}__{k}"            if args.channels:                order = [s for s, _ in suff]                cmap = dict(zip([x.strip() for x in args.channels.split(",")], order))                bf_s, g_s, r_s = cmap["bf"], cmap["green"], cmap["red"]            else:                means = {s: _mean_gray(p) for s, p in suff}                order = sorted(means, key=means.get, reverse=True)   # brightest -> darkest                bf_s, g_s, r_s = order[0], order[1], order[2]            rows.append({                "id": sid, "series": series.name,                "bf": f"{series.name}/{suff[bf_s][1].name}",                "green": f"{series.name}/{suff[g_s][1].name}",                "red": f"{series.name}/{suff[r_s][1].name}",            })    df = pd.DataFrame(rows)    # Slide-stratified split: 70/15/15 train/val/test (whole series stay together)    rng = np.random.default_rng(42)    series = sorted(df["series"].unique())    rng.shuffle(series)    n_tr = max(1, int(round(len(series) * 0.70)))    n_va = max(1, int(round(len(series) * 0.15)))    split_of = {s: "train" for s in series[:n_tr]}    split_of.update({s: "val" for s in series[n_tr:n_tr + n_va]})    split_of.update({s: "test" for s in series[n_tr + n_va:]})    df["split"] = df["series"].map(split_of)    print(f"paired {len(df)} fields; channel auto-map used "          f"(set --channels if wrong). Split: {df['split'].value_counts().to_dict()}")    print(df.head(3).to_string())    df.to_csv(args.out, index=False)    print(f"wrote {args.out}")if __name__ == "__main__":    main()

In [ ]:
%%writefile data/download_rxrx3_core.py"""Download RxRx3-core metadata + OpenPhenom embeddings (Module C extension).Repo: recursionpharma/rxrx3-core on HuggingFace (CC BY 4.0).Usage:  python data/download_rxrx3_core.py --out data/rxrx3"""from __future__ import annotationsimport argparsefrom pathlib import Pathdef main():    ap = argparse.ArgumentParser()    ap.add_argument("--out", default="data/rxrx3")    ap.add_argument("--metadata-only", action="store_true")    args = ap.parse_args()    from huggingface_hub import hf_hub_download    out = Path(args.out)    out.mkdir(parents=True, exist_ok=True)    files = ["metadata_rxrx3_core.csv"]    if not args.metadata_only:        files.append("OpenPhenom_rxrx3_core_embeddings.parquet")    for f in files:        p = hf_hub_download("recursionpharma/rxrx3-core", f, repo_type="dataset")        print("downloaded", p)    print(f"done -> {out}")if __name__ == "__main__":    main()

## Install + data

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-image", "tqdm", "pyarrow", "huggingface_hub"], check=True)

In [ ]:
!python data/download_caco2_fast.py --out data/caco2/raw
!python data/make_caco2_pairs.py --root data/caco2/raw --out data/caco2/pairs.csv

## Module A — train (GPU, 60 epochs, base64/depth4 + perceptual)

In [ ]:
!python -m labelfree.train --config configs/train_caco2.yaml --out runs/caco2

## Module A — MC-dropout inference + evaluation

In [ ]:
!python -m labelfree.infer --config configs/infer.yaml --ckpt runs/caco2/best.ckpt \
    --csv data/caco2/pairs.csv --root data/caco2/raw --split test \
    --crop 256 --mc-samples 3 --base 64 --depth 4 --out runs/caco2/infer
!python -m labelfree.evaluate_cli --csv data/caco2/pairs.csv --root data/caco2/raw \
    --pred-dir runs/caco2/infer --split test --crop 256 --out runs/caco2/eval
import json
print(json.dumps(json.load(open("runs/caco2/eval/summary.json")), indent=2))

## Module B — label-free viability vs. real stain

In [ ]:
!python scripts/viability_agreement.py --run runs/caco2 --out report/assets!python scripts/make_figures.py --run runs/caco2 --out report/assets --n 4

## Module C — RxRx3-core dose-response phenomics (~5 min, embeddings only)

In [ ]:
!python scripts/rxrx3_dose_response.py --out runs/rxrx3
import json
fits = json.load(open("runs/rxrx3/dose_response_fits.json"))
print("compounds fitted:", len(fits))
for name, f in list(fits.items())[:3]:
    print(f"  {name}: EC50={f['EC50']:.3f} uM  R2={f['R2']:.3f}")

## Outputs (download)- `runs/caco2/eval/summary.json` per-channel PSNR/SSIM- `runs/caco2/infer/*_pred_*.png` virtual stains + uncertainty- `runs/caco2/viability_agreement.json` Module B- `runs/rxrx3/dose_response_fits.json` Module C (EC50/Emax/R²)- `runs/caco2/best.ckpt` model